# FMP Cache Daily Updater

## 목적
FMP 로컬 캐시를 최신 상태로 유지합니다.

---

## 사용 가이드

### 환경 설정
- `FMP_API_KEY` 환경변수가 설정되어 있어야 합니다.
- PyCharm: `Run > Edit Configurations > Environment Variables`에서 설정

### 플랜별 실행 방법

| 플랜 | 방법 | 셀 실행 순서 |
|---|---|---|
| **Free (250 calls/day)** | Day 1 + Day 2 격일 실행 | Cell 0 → Cell 2 (Day1) 또는 Cell 3 (Day2) |
| **Premium** | 전체 한번에 | Cell 0 → Cell 4 |
| **신규 종목 보충** | 누락 종목 OHLCV backfill | Cell 0 → Cell 5 |
| **Financials 보충** | 누락 종목 재무제표 다운로드 | Cell 0 → Cell 6 |
| **Marketcap 보충** | 누락 종목 시가총액 히스토리 | Cell 0 → Cell 7 |
| **상태 점검** | 캐시 현황 확인 | Cell 0 → Cell 1 |

---

### Free Plan 250 calls 계산

```
Universe refresh: 1 call (캐시 만료시 1회, 보통 1회/일)
OHLCV delta:      503 tickers → Day1 249 + Day2 253 (2일 순환)
→ 총 250 calls/day 안에서 처리 가능
```

> ⚠️ **Financials / Marketcap / Backfill은 프리미엄 플랜 필요**  
> 재무제표(6 statements × 503 tickers = ~3,018 calls)와 Historical Marketcap은  
> 무료 플랜으로는 다운로드 불가능합니다. 프리미엄 플랜 유효 기간 중에 미리 수집해두세요.

---

### 데이터 갱신 주기 권고

| 데이터 | 권고 주기 | 비고 |
|---|---|---|
| OHLCV | **매일** | Free: 2일 순환, Premium: 매일 전체 |
| Marketcap | **주 1회** | Historical은 분기 1회로 충분 |
| Financials | **분기 1회** (실적 발표 후) | Q1: 5월, Q2: 8월, Q3: 11월, Q4: 3월 |
| Universe | **자동** (TTL 24h) | 새 종목 편입 감지 → backfill 필요 |


In [1]:
# Cell 0 — 초기화 (항상 먼저 실행)
import sys
sys.path.insert(0, "/Users/shin-il/PyCharmMiscProject/0316-")

import os
import json
import time
import requests
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional, Any

# ── 설정값 (필요시 수정) ─────────────────────────────────────
CACHE_ROOT   = "/Users/shin-il/Documents/my stock/cache_fmp_c2_1"
FMP_BASE     = "https://financialmodelingprep.com"
CHUNK_YEARS  = 5
OHLCV_COLS   = ["date", "open", "high", "low", "close", "volume"]

FREE_PLAN_LIMIT = 250   # calls/day
DAY1_END_IDX    = 249   # 0~248 → Day 1 (249 tickers + 1 universe = 250)

# ── 유틸 함수 ────────────────────────────────────────────────
def get_api_key() -> str:
    key = os.environ.get("FMP_API_KEY", "").strip()
    if not key:
        raise RuntimeError("FMP_API_KEY 환경변수가 설정되지 않았습니다.")
    return key

def safe_fn(t: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in str(t))

def fmp_get(url: str, params: dict, timeout: int = 30) -> Any:
    params = {**params, "apikey": get_api_key()}
    r = requests.get(url, params=params, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    return r.json()

def write_ohlcv(df: pd.DataFrame, path: str) -> None:
    df = df[[c for c in OHLCV_COLS if c in df.columns]].copy()
    dates = pa.array(pd.to_datetime(df["date"]).astype("datetime64[ns]"))
    arrays, names = [dates], ["date"]
    for c in ["open","high","low","close","volume"]:
        if c in df.columns:
            arrays.append(pa.array(pd.to_numeric(df[c], errors="coerce").fillna(0).astype("float64")))
            names.append(c)
    pq.write_table(pa.table(dict(zip(names, arrays))), path)

def normalize_ohlcv(j: list) -> pd.DataFrame:
    df = pd.DataFrame(j)
    if df.empty:
        return pd.DataFrame(columns=OHLCV_COLS)
    for old, new in [("Date","date"),("Open","open"),("High","high"),("Low","low"),("Close","close"),("Volume","volume")]:
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.tz_localize(None)
    df = df.dropna(subset=["date"]).sort_values("date").drop_duplicates("date", keep="last")
    for c in OHLCV_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[OHLCV_COLS].copy()

def get_last_ohlcv_date(ticker: str) -> Optional[datetime]:
    tdir = os.path.join(CACHE_ROOT, "ohlcv", safe_fn(ticker))
    if not os.path.isdir(tdir):
        return None
    chunks = sorted(f for f in os.listdir(tdir) if f.endswith(".parquet"))
    if not chunks:
        return None
    try:
        df = pd.read_parquet(os.path.join(tdir, chunks[-1]))
        return pd.Timestamp(df["date"].max()).to_pydatetime()
    except Exception:
        return None

def update_ohlcv_delta(ticker: str, target_date: datetime, rate_sleep: float = 0.2) -> dict:
    """기존 최신 chunk에 delta row들을 append합니다."""
    tdir = os.path.join(CACHE_ROOT, "ohlcv", safe_fn(ticker))
    os.makedirs(tdir, exist_ok=True)

    last = get_last_ohlcv_date(ticker)
    if last is None:
        start = datetime(2016, 1, 1)
        mode = "backfill"
    else:
        start = last + timedelta(days=1)
        mode = "delta"

    if start.date() > target_date.date():
        return {"ticker": ticker, "mode": "skip", "rows": 0, "calls": 0}

    url = f"{FMP_BASE}/stable/historical-price-eod/full"
    j = fmp_get(url, {"symbol": ticker, "from": start.strftime("%Y-%m-%d"), "to": target_date.strftime("%Y-%m-%d")}, timeout=60)

    if isinstance(j, dict):
        return {"ticker": ticker, "mode": mode, "error": str(j)[:80], "calls": 1}

    new_df = normalize_ohlcv(j if isinstance(j, list) else [])
    if new_df.empty:
        return {"ticker": ticker, "mode": "skip", "rows": 0, "calls": 1}

    # 기존 최신 chunk에 merge
    chunks = sorted(f for f in os.listdir(tdir) if f.endswith(".parquet"))
    if chunks:
        latest_path = os.path.join(tdir, chunks[-1])
        try:
            existing = pd.read_parquet(latest_path)
        except Exception:
            existing = pd.DataFrame(columns=OHLCV_COLS)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined["date"] = pd.to_datetime(combined["date"], errors="coerce")
        combined = combined.dropna(subset=["date"]).sort_values("date").drop_duplicates("date", keep="last")
        combined = combined[OHLCV_COLS].copy()
        write_ohlcv(combined, latest_path)
    else:
        # 새 파일: 2026_2027.parquet 생성
        new_path = os.path.join(tdir, "2026_2027.parquet")
        write_ohlcv(new_df, new_path)

    return {"ticker": ticker, "mode": mode, "rows": len(new_df), "calls": 1}

def load_sp500_tickers(force_refresh: bool = False) -> List[str]:
    uni_dir = os.path.join(CACHE_ROOT, "universe")
    os.makedirs(uni_dir, exist_ok=True)
    path = os.path.join(uni_dir, "sp500_tickers.json")
    if not force_refresh and os.path.exists(path):
        mtime = datetime.fromtimestamp(os.path.getmtime(path))
        if (datetime.now() - mtime) < timedelta(hours=24):
            with open(path) as f:
                data = json.load(f)
            tickers = data.get("tickers", [])
            print(f"[Universe] 캐시 사용: {len(tickers)}개 (마지막 갱신: {mtime:%Y-%m-%d %H:%M})")
            return tickers
    url = f"{FMP_BASE}/stable/sp500-constituent"
    j = fmp_get(url, {})
    tickers = sorted(set(row.get("symbol","") for row in j if row.get("symbol")))
    with open(path, "w") as f:
        json.dump({"fetched_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "ttl_days": 7, "tickers": tickers, "source": url}, f, indent=2)
    print(f"[Universe] 갱신 완료: {len(tickers)}개")
    return tickers

print("✅ Cell 0 초기화 완료")
print(f"   Cache root: {CACHE_ROOT}")
print(f"   FMP API Key: {'설정됨' if os.environ.get('FMP_API_KEY') else '❌ 미설정'}")


✅ Cell 0 초기화 완료
   Cache root: /Users/shin-il/Documents/my stock/cache_fmp_c2_1
   FMP API Key: 설정됨


In [ ]:
# Cell 1 — 캐시 상태 점검 (API 호출 없음)
# 실행 전: Cell 0 실행 필요

target_date = datetime.now()
cutoff_str  = (target_date - timedelta(days=5)).strftime("%Y-%m-%d")

tickers = load_sp500_tickers()

missing_ohlcv = []
stale_ohlcv   = []
current_ohlcv = []

for t in tickers:
    last = get_last_ohlcv_date(t)
    if last is None:
        missing_ohlcv.append(t)
    elif last.strftime("%Y-%m-%d") < cutoff_str:
        stale_ohlcv.append((t, last.strftime("%Y-%m-%d")))
    else:
        current_ohlcv.append(t)

fin_missing = [t for t in tickers if not os.path.isdir(os.path.join(CACHE_ROOT, "financials", safe_fn(t))) or len(os.listdir(os.path.join(CACHE_ROOT, "financials", safe_fn(t)))) < 6]
mcap_missing = [t for t in tickers if not os.path.exists(os.path.join(CACHE_ROOT, "marketcap", f"{safe_fn(t)}.parquet"))]

print("=" * 60)
print("캐시 상태 리포트")
print("=" * 60)
print(f"기준 날짜:         {target_date.strftime('%Y-%m-%d')}")
print(f"총 S&P500 종목:    {len(tickers)}")
print(f"OHLCV 최신:        {len(current_ohlcv)} / {len(tickers)}")
print(f"OHLCV 오래됨:      {len(stale_ohlcv)}")
print(f"OHLCV 없음:        {len(missing_ohlcv)}")
print(f"Financials 없음:   {len(fin_missing)}")
print(f"Marketcap 없음:    {len(mcap_missing)}")

if missing_ohlcv:
    print(f"\n❌ OHLCV 없음: {missing_ohlcv}")
    print("   → Cell 5 (Backfill) 실행 필요 [프리미엄 불필요]")
if stale_ohlcv:
    print(f"\n⚠️  OHLCV 오래됨 (상위 10개):")
    for t, d in sorted(stale_ohlcv, key=lambda x: x[1])[:10]:
        print(f"   {t}: {d}")
    print("   → Cell 2/3 (Free) 또는 Cell 4 (Premium) 실행")
if fin_missing:
    print(f"\n❌ Financials 없음: {fin_missing}")
    print("   → Cell 6 실행 필요 [프리미엄 필요]")
if mcap_missing:
    print(f"\n❌ Marketcap 없음: {mcap_missing}")
    print("   → Cell 7 실행 필요 [프리미엄 필요]")
if not missing_ohlcv and not stale_ohlcv and not fin_missing and not mcap_missing:
    print("\n✅ 모든 캐시 최신 상태입니다!")
print("=" * 60)


In [ ]:
# Cell 2 — [Free Plan] OHLCV Day 1 업데이트 (종목 0~248, ~249 calls)
# 실행 전: Cell 0 실행 필요
# 격일 실행: 오늘 Day 1, 내일 Day 2 (Cell 3)
# Free Plan: 1일 최대 250 calls (universe 1 + OHLCV 249)

TARGET_DATE = datetime.now()           # 오늘까지 업데이트
RATE_SLEEP  = 0.25                     # Free Plan: 초당 최대 4 calls

tickers = load_sp500_tickers()
batch = tickers[:DAY1_END_IDX]         # 0 ~ 248번 종목

print(f"[Day 1] {len(batch)}개 종목 OHLCV 업데이트 시작 ({batch[0]} ~ {batch[-1]})")
print(f"예상 API 호출: {len(batch)+1} / {FREE_PLAN_LIMIT} (universe 1 포함)")
print()

results = {"updated": 0, "skipped": 0, "errors": 0, "calls": 1}  # 1 = universe
for i, ticker in enumerate(batch):
    try:
        r = update_ohlcv_delta(ticker, TARGET_DATE, RATE_SLEEP)
        results["calls"] += r.get("calls", 0)
        if r.get("error"):
            results["errors"] += 1
            print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ❌ {r['error']}")
        elif r["mode"] == "skip":
            results["skipped"] += 1
            if (i+1) % 50 == 0:
                print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ⏭️  skip (이미 최신)")
        else:
            results["updated"] += 1
            print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ✅ {r['mode']} +{r['rows']} rows")
        time.sleep(RATE_SLEEP)
    except Exception as e:
        results["errors"] += 1
        print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ❌ {e}")

print()
print("=" * 50)
print(f"Day 1 완료: updated={results['updated']} skipped={results['skipped']} errors={results['errors']} total_calls={results['calls']}")
print("=" * 50)


In [ ]:
# Cell 3 — [Free Plan] OHLCV Day 2 업데이트 (종목 249~502)
# 실행 전: Cell 0 실행 필요
# Day 1 다음날 실행

TARGET_DATE = datetime.now()
RATE_SLEEP  = 0.25

tickers = load_sp500_tickers()
batch = tickers[DAY1_END_IDX:]         # 249번 이후 종목

# Free Plan 한도 보호 (universe 포함 250)
max_batch = FREE_PLAN_LIMIT - 1
if len(batch) > max_batch:
    print(f"⚠️  Free Plan 한도 보호: {len(batch)} → {max_batch}개로 제한")
    batch = batch[:max_batch]

print(f"[Day 2] {len(batch)}개 종목 OHLCV 업데이트 시작 ({batch[0]} ~ {batch[-1]})")
print(f"예상 API 호출: {len(batch)+1} / {FREE_PLAN_LIMIT} (universe 1 포함)")
print()

results = {"updated": 0, "skipped": 0, "errors": 0, "calls": 1}
for i, ticker in enumerate(batch):
    try:
        r = update_ohlcv_delta(ticker, TARGET_DATE, RATE_SLEEP)
        results["calls"] += r.get("calls", 0)
        if r.get("error"):
            results["errors"] += 1
            print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ❌ {r['error']}")
        elif r["mode"] == "skip":
            results["skipped"] += 1
            if (i+1) % 50 == 0:
                print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ⏭️  skip")
        else:
            results["updated"] += 1
            print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ✅ {r['mode']} +{r['rows']} rows")
        time.sleep(RATE_SLEEP)
    except Exception as e:
        results["errors"] += 1
        print(f"  [{i+1:3d}/{len(batch)}] {ticker:8s} ❌ {e}")

print()
print("=" * 50)
print(f"Day 2 완료: updated={results['updated']} skipped={results['skipped']} errors={results['errors']} total_calls={results['calls']}")
print("=" * 50)


In [2]:
# Cell 4 — [Premium Plan] OHLCV 전체 업데이트 (503 tickers 한번에)
# 실행 전: Cell 0 실행 필요

TARGET_DATE = datetime.now()
RATE_SLEEP  = 0.08   # Premium: 더 빠른 속도

tickers = load_sp500_tickers()
print(f"[Premium] {len(tickers)}개 종목 전체 OHLCV 업데이트 시작")
print(f"예상 소요시간: ~{len(tickers) * RATE_SLEEP / 60:.1f}분")
print()

results = {"updated": 0, "skipped": 0, "errors": 0, "calls": 1}
for i, ticker in enumerate(tickers):
    try:
        r = update_ohlcv_delta(ticker, TARGET_DATE, RATE_SLEEP)
        results["calls"] += r.get("calls", 0)
        if r.get("error"):
            results["errors"] += 1
            print(f"  [{i+1:3d}/{len(tickers)}] {ticker:8s} ❌ {r['error']}")
        elif r["mode"] == "skip":
            results["skipped"] += 1
            if (i+1) % 100 == 0:
                print(f"  [{i+1:3d}/{len(tickers)}] ... (skipped)")
        else:
            results["updated"] += 1
            print(f"  [{i+1:3d}/{len(tickers)}] {ticker:8s} ✅ +{r['rows']} rows")
        time.sleep(RATE_SLEEP)
    except Exception as e:
        results["errors"] += 1
        print(f"  [{i+1:3d}/{len(tickers)}] {ticker:8s} ❌ {e}")

print()
print("=" * 50)
print(f"전체 완료: updated={results['updated']} skipped={results['skipped']} errors={results['errors']} total_calls={results['calls']}")
print("=" * 50)


[Universe] 갱신 완료: 503개
[Premium] 503개 종목 전체 OHLCV 업데이트 시작
예상 소요시간: ~0.7분

  [  1/503] A        ✅ +2 rows
  [  2/503] AAPL     ✅ +2 rows
  [  3/503] ABBV     ✅ +2 rows
  [  4/503] ABNB     ✅ +2 rows
  [  5/503] ABT      ✅ +2 rows
  [  6/503] ACGL     ✅ +2 rows
  [  7/503] ACN      ✅ +2 rows
  [  8/503] ADBE     ✅ +2 rows
  [  9/503] ADI      ✅ +2 rows
  [ 10/503] ADM      ✅ +2 rows
  [ 11/503] ADP      ✅ +2 rows
  [ 12/503] ADSK     ✅ +2 rows
  [ 13/503] AEE      ✅ +2 rows
  [ 14/503] AEP      ✅ +2 rows
  [ 15/503] AES      ✅ +2 rows
  [ 16/503] AFL      ✅ +2 rows
  [ 17/503] AIG      ✅ +2 rows
  [ 18/503] AIZ      ✅ +2 rows
  [ 19/503] AJG      ✅ +2 rows
  [ 20/503] AKAM     ✅ +2 rows
  [ 21/503] ALB      ✅ +2 rows
  [ 22/503] ALGN     ✅ +2 rows
  [ 23/503] ALL      ✅ +2 rows
  [ 24/503] ALLE     ✅ +2 rows
  [ 25/503] AMAT     ✅ +2 rows
  [ 26/503] AMCR     ✅ +2 rows
  [ 27/503] AMD      ✅ +2 rows
  [ 28/503] AME      ✅ +2 rows
  [ 29/503] AMGN     ✅ +2 rows
  [ 30/503] AMP      ✅ +2 r

In [ ]:
# Cell 5 — OHLCV Backfill (신규 종목 / 누락 종목 전체 다운로드)
# 실행 전: Cell 0 실행 필요
# 프리미엄 플랜에서 실행 권장 (종목당 multiple calls)
# 특정 종목만 backfill 하려면 BACKFILL_TICKERS에 지정

TARGET_DATE     = datetime.now()
BACKFILL_START  = datetime(2016, 1, 1)   # 기본 2016년부터
RATE_SLEEP      = 0.08

# None이면 자동으로 누락 종목 탐지 / 직접 지정도 가능
BACKFILL_TICKERS = None  # 예: ["COHR", "VRT", "LITE", "SATS"]

tickers = load_sp500_tickers()

if BACKFILL_TICKERS is None:
    BACKFILL_TICKERS = [t for t in tickers if get_last_ohlcv_date(t) is None]

if not BACKFILL_TICKERS:
    print("✅ 누락 종목 없음 - backfill 불필요")
else:
    print(f"[Backfill] {len(BACKFILL_TICKERS)}개 종목: {BACKFILL_TICKERS}")
    print(f"예상 API 호출: ~{len(BACKFILL_TICKERS) * 3} (종목당 ~3 chunks)")
    print()

    for i, ticker in enumerate(BACKFILL_TICKERS):
        tdir = os.path.join(CACHE_ROOT, "ohlcv", safe_fn(ticker))
        os.makedirs(tdir, exist_ok=True)
        url = f"{FMP_BASE}/stable/historical-price-eod/full"
        total_rows = 0
        y0, y1 = BACKFILL_START.year, TARGET_DATE.year
        for a in range(y0, y1 + 1, CHUNK_YEARS):
            b = a + CHUNK_YEARS
            fn = f"{a}_{b}.parquet"
            path = os.path.join(tdir, fn)
            from_dt = datetime(a, 1, 1)
            to_dt = min(datetime(b - 1, 12, 31), TARGET_DATE)
            try:
                j = fmp_get(url, {"symbol": ticker, "from": from_dt.strftime("%Y-%m-%d"), "to": to_dt.strftime("%Y-%m-%d")}, timeout=60)
                df = normalize_ohlcv(j if isinstance(j, list) else [])
                if not df.empty:
                    write_ohlcv(df, path)
                    total_rows += len(df)
                time.sleep(RATE_SLEEP)
            except Exception as e:
                print(f"  {ticker} chunk {a}_{b}: ❌ {e}")
        print(f"  [{i+1}/{len(BACKFILL_TICKERS)}] {ticker}: ✅ backfill 완료 ({total_rows} rows)")

    print("\n[Backfill] 완료")


In [ ]:
# Cell 6 — Financials 다운로드 (income / balance / cashflow)
# ⚠️  프리미엄 플랜 필요 (종목당 6 calls = quarter+annual × 3 statements)
# 권고: 분기 실적 시즌 후 1회 실행 (Q1: 5월, Q2: 8월, Q3: 11월, Q4: 3월)
# 실행 전: Cell 0 실행 필요

RATE_SLEEP = 0.12
LIMIT      = 100   # FMP limit parameter (최대 100개 분기)
OVERWRITE  = True  # 기존 파일 덮어쓰기

# None이면 누락 종목만 자동 선택 / 직접 지정 가능
FIN_TICKERS = None  # 예: ["COHR", "LITE", "SATS", "VRT"]

tickers = load_sp500_tickers()

if FIN_TICKERS is None:
    fin_dir = os.path.join(CACHE_ROOT, "financials")
    FIN_TICKERS = [t for t in tickers if not os.path.isdir(os.path.join(fin_dir, safe_fn(t))) or len(os.listdir(os.path.join(fin_dir, safe_fn(t)))) < 6]

if not FIN_TICKERS:
    print("✅ 모든 종목의 financials 캐시가 이미 존재합니다.")
else:
    print(f"[Financials] {len(FIN_TICKERS)}개 종목: {FIN_TICKERS}")
    print(f"예상 API 호출: {len(FIN_TICKERS) * 6} (종목당 6 calls)")
    print()

    FIN_ENDPOINTS = {
        "income_quarter":   "/stable/income-statement",
        "income_annual":    "/stable/income-statement",
        "balance_quarter":  "/stable/balance-sheet-statement",
        "balance_annual":   "/stable/balance-sheet-statement",
        "cashflow_quarter": "/stable/cash-flow-statement",
        "cashflow_annual":  "/stable/cash-flow-statement",
    }
    FIN_PERIODS = {
        "income_quarter": "quarter", "income_annual": "annual",
        "balance_quarter": "quarter", "balance_annual": "annual",
        "cashflow_quarter": "quarter", "cashflow_annual": "annual",
    }

    errors = 0
    for i, ticker in enumerate(FIN_TICKERS):
        tdir = os.path.join(CACHE_ROOT, "financials", safe_fn(ticker))
        os.makedirs(tdir, exist_ok=True)
        ticker_ok = True
        for key, endpoint in FIN_ENDPOINTS.items():
            path = os.path.join(tdir, f"{key}.parquet")
            if not OVERWRITE and os.path.exists(path):
                continue
            period = FIN_PERIODS[key]
            try:
                j = fmp_get(f"{FMP_BASE}{endpoint}", {"symbol": ticker, "period": period, "limit": LIMIT})
                df = pd.DataFrame(j if isinstance(j, list) else [])
                if not df.empty:
                    if "date" not in df.columns:
                        for c in ["Date", "DATE", "filingDate", "period"]:
                            if c in df.columns:
                                df = df.rename(columns={c: "date"})
                                break
                    df["date"] = pd.to_datetime(df.get("date", pd.Series()), errors="coerce").dt.tz_localize(None)
                    df.to_parquet(path, index=False)
                time.sleep(RATE_SLEEP)
            except Exception as e:
                ticker_ok = False
                errors += 1
                print(f"  {ticker}/{key}: ❌ {e}")
        status = "✅" if ticker_ok else "⚠️ "
        print(f"  [{i+1}/{len(FIN_TICKERS)}] {ticker}: {status} financials 저장")

    print(f"\n[Financials] 완료 (errors={errors})")


In [ ]:
# Cell 7 — Marketcap 히스토리 다운로드
# ⚠️  프리미엄 플랜 필요
# 실행 전: Cell 0 실행 필요

RATE_SLEEP   = 0.12
MCAP_LIMIT   = 5000   # 최대 조회 개수
OVERWRITE    = True

# None이면 누락 종목만 / 직접 지정 가능
MCAP_TICKERS = None  # 예: ["COHR", "LITE", "SATS", "VRT"]

tickers = load_sp500_tickers()

if MCAP_TICKERS is None:
    mcap_dir = os.path.join(CACHE_ROOT, "marketcap")
    MCAP_TICKERS = [t for t in tickers if not os.path.exists(os.path.join(mcap_dir, f"{safe_fn(t)}.parquet"))]

if not MCAP_TICKERS:
    print("✅ 모든 종목의 marketcap 캐시가 이미 존재합니다.")
else:
    print(f"[Marketcap] {len(MCAP_TICKERS)}개 종목: {MCAP_TICKERS}")
    print(f"예상 API 호출: {len(MCAP_TICKERS)} calls")
    print()

    mcap_dir = os.path.join(CACHE_ROOT, "marketcap")
    os.makedirs(mcap_dir, exist_ok=True)
    errors = 0

    for i, ticker in enumerate(MCAP_TICKERS):
        path = os.path.join(mcap_dir, f"{safe_fn(ticker)}.parquet")
        if not OVERWRITE and os.path.exists(path):
            print(f"  [{i+1}/{len(MCAP_TICKERS)}] {ticker}: ⏭️  skip (이미 존재)")
            continue
        try:
            j = fmp_get(f"{FMP_BASE}/stable/historical-market-capitalization",
                        {"symbol": ticker, "limit": MCAP_LIMIT})
            df = pd.DataFrame(j if isinstance(j, list) else [])
            if not df.empty:
                if "date" not in df.columns:
                    for c in ["Date", "DATE"]:
                        if c in df.columns:
                            df = df.rename(columns={c: "date"})
                            break
                df["date"] = pd.to_datetime(df.get("date", pd.Series()), errors="coerce").dt.tz_localize(None)
                if "symbol" not in df.columns:
                    df["symbol"] = ticker
                df = df.sort_values("date").drop_duplicates("date", keep="last")
                df.to_parquet(path, index=False)
                print(f"  [{i+1}/{len(MCAP_TICKERS)}] {ticker}: ✅ {len(df)} rows")
            else:
                print(f"  [{i+1}/{len(MCAP_TICKERS)}] {ticker}: ⚠️  응답 비어있음")
            time.sleep(RATE_SLEEP)
        except Exception as e:
            errors += 1
            print(f"  [{i+1}/{len(MCAP_TICKERS)}] {ticker}: ❌ {e}")

    print(f"\n[Marketcap] 완료 (errors={errors})")
